In [0]:
import mlflow
mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="test_installation"):
    mlflow.log_param("test", 1)
    mlflow.log_metric("pr_auc", 0.5)

print("MLflow OK, version", mlflow.__version__)

In [0]:
from pyspark.sql import functions as F

SOURCE = "/Volumes/olist_dev/raw/source_olist"

FICHIERS = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

def lire_csv(nom_fichier):
    return (
        spark.read
        .option("header", True)
        .option("multiLine", True)   # avis sur plusieurs lignes
        .option("escape", '"')       # guillemets doublés dans le texte
        .option("inferSchema", True) # OK pour explorer, interdit en Bronze
        .csv(f"{SOURCE}/{nom_fichier}")
    )

dfs = {nom: lire_csv(fichier) for nom, fichier in FICHIERS.items()}

for nom, df in dfs.items():
    print(f"{nom:<22} {df.count():>10,} lignes  {len(df.columns):>3} colonnes")

In [0]:
CLES = {
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id"],
    "customers": ["customer_id"],
    "sellers": ["seller_id"],
    "products": ["product_id"],
    "category_translation": ["product_category_name"],
}

for nom, cle in CLES.items():
    df = dfs[nom]
    total = df.count()
    distinctes = df.select(*cle).distinct().count()
    nulles = df.filter(" OR ".join(f"{c} IS NULL" for c in cle)).count()
    statut = "OK" if total == distinctes and nulles == 0 else "⚠️ À TRAITER"
    print(f"{nom:<22} total={total:>8,}  distinctes={distinctes:>8,}  nulles={nulles:>4}  {statut}")

In [0]:
for nom, df in dfs.items():
    print(f"\n{'=' * 20} {nom} ({df.count():,} lignes) {'=' * 20}")
    df.printSchema()
    display(df.limit(5))

In [0]:
orders = dfs["orders"]

display(orders.groupBy("order_status").count().orderBy(F.desc("count")))

livrees = orders.filter(
    (F.col("order_status") == "delivered")
    & F.col("order_delivered_customer_date").isNotNull()
)

cible = livrees.withColumn(
    "en_retard",
    (F.to_date("order_delivered_customer_date") > F.to_date("order_estimated_delivery_date")).cast("int"),
)

display(cible.agg(
    F.count("*").alias("commandes_livrees"),
    F.sum("en_retard").alias("retards"),
    F.round(F.avg("en_retard") * 100, 2).alias("taux_retard_pct"),
))

In [0]:
par_mois = (
    cible.groupBy(F.date_trunc("month", "order_purchase_timestamp").alias("mois"))
    .agg(
        F.count("*").alias("commandes"),
        F.round(F.avg("en_retard") * 100, 1).alias("taux_retard_pct"),
    )
    .orderBy("mois")
)
display(par_mois)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
fin = (
    orders.filter(F.col("order_purchase_timestamp") >= "2018-06-01")
    .groupBy(F.date_trunc("week", "order_purchase_timestamp").alias("semaine"))
    .agg(
        F.count("*").alias("commandes"),
        F.round(F.avg((F.col("order_status") == "delivered").cast("int")) * 100, 1).alias("pct_livrees"),
    )
    .orderBy("semaine")
)
display(fin)

In [0]:
controles = {
    "approbation manquante": F.col("order_approved_at").isNull(),
    "approuvée avant l'achat": F.col("order_approved_at") < F.col("order_purchase_timestamp"),
    "transporteur avant approbation": F.col("order_delivered_carrier_date") < F.col("order_approved_at"),
    "livrée avant transporteur": F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date"),
    "delivered sans date de livraison": (F.col("order_status") == "delivered") & F.col("order_delivered_customer_date").isNull(),
    "date de livraison mais pas delivered": (F.col("order_status") != "delivered") & F.col("order_delivered_customer_date").isNotNull(),
}

total = orders.count()
for nom, condition in controles.items():
    n = orders.filter(condition).count()
    print(f"{nom:<40} {n:>7,}  ({n / total * 100:.2f} %)")

In [0]:
signaux = (
    cible
    .join(dfs["customers"].select("customer_id", "customer_state"), "customer_id")
    .withColumn("delai_promis_j", F.datediff("order_estimated_delivery_date", "order_purchase_timestamp"))
)

# Le retard selon l'État du client
display(
    signaux.groupBy("customer_state")
    .agg(F.count("*").alias("commandes"), F.round(F.avg("en_retard") * 100, 1).alias("taux_retard_pct"))
    .orderBy(F.desc("taux_retard_pct"))
)

# Le retard selon le délai promis au client
display(
    signaux.groupBy((F.floor(F.col("delai_promis_j") / 7) * 7).alias("delai_promis_j"))
    .agg(F.count("*").alias("commandes"), F.round(F.avg("en_retard") * 100, 1).alias("taux_retard_pct"))
    .orderBy("delai_promis_j")
)